# Runs complémentaires Fashion-MNIST (Colab, GPU)

1. **Confirmation du régime sur des graines neuves** : sans dropout et dropout classique, graines 5 à 9, sur MLP largeur 16 et CNN largeur 4 (50 epochs). Le régime a été choisi avec les graines 0 et 1, qui sont aussi dans la grille : on le revérifie sur des graines qui n'ont servi à rien. 20 runs.
2. **Durée plus courte** : early dropout pendant 1 epoch sur 50 (frac 0.02), graines 0 à 4, mêmes modèles. Notre meilleur point était la durée la plus courte testée (2 epochs), on regarde si c'est encore mieux en dessous. 10 runs.

In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")
%cd /content/drive/MyDrive/early-dropout
# !git stash
!git pull

In [ ]:
import os, glob, importlib
import torch
from torchvision import datasets

import config, models, train
importlib.reload(config); importlib.reload(models); importlib.reload(train)
from config import BATCH, P
from models import mlp, cnn
from train import entrainer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device, "| p =", P, "| batch =", BATCH)

In [ ]:
tr = datasets.FashionMNIST("./data", train=True, download=True)
te = datasets.FashionMNIST("./data", train=False, download=True)
Xtr, Xte = tr.data.float().unsqueeze(1) / 255, te.data.float().unsqueeze(1) / 255
ytr, yte = tr.targets, te.targets

# même split et même normalisation que dans fashion_mnist.ipynb
perm = torch.randperm(len(Xtr), generator=torch.Generator().manual_seed(0))
Xva, yva = Xtr[perm[:10000]], ytr[perm[:10000]]
Xtr, ytr = Xtr[perm[10000:]], ytr[perm[10000:]]
mu, sd = Xtr.mean(), Xtr.std()
Xtr, Xva, Xte = (Xtr - mu) / sd, (Xva - mu) / sd, (Xte - mu) / sd

images = {k: v.to(device) for k, v in dict(Xtr=Xtr, ytr=ytr, Xva=Xva, yva=yva, Xte=Xte, yte=yte).items()}   # CNN
vecteurs = {k: (v.flatten(1) if k.startswith("X") else v) for k, v in images.items()}                        # MLP

In [ ]:
# même fonction que dans fashion_mnist.ipynb : un run = un CSV, sauté s'il existe déjà
CADRES = {
    "mlp_fmnist": (lambda L: mlp(784, 10, couche_cache=L), vecteurs),
    "cnn_fmnist": (lambda L: cnn(1, 10, largeur=L), images),
}

def fichier(cadre, L, E, frac, seed, dossier="resultats"):
    return f"{dossier}/{cadre}_w{L}_e{E}_frac{frac}_s{seed}.csv"

def lancer(cadre, L, E, frac, seed, p=P, dossier="resultats"):
    f = fichier(cadre, L, E, frac, seed, dossier)
    if os.path.exists(f):
        return
    creer, data = CADRES[cadre]
    torch.manual_seed(seed)
    model, opt = creer(L)
    model.to(device)
    hist = entrainer(model, opt, data, E, t_early=round(frac * E), p=p, batch=BATCH)
    hist.assign(cadre=cadre, width=L, epochs=E, frac=frac, seed=seed).to_csv(f, index=False)
    print(f, " val acc", round(hist.val_acc.tail(5).mean(), 4))

## 1. Confirmation du régime sur graines neuves (20 runs)

In [ ]:
GRAINES_NEUVES = [5, 6, 7, 8, 9]
for seed in GRAINES_NEUVES:
    for frac in [0, 1]:                        # sans dropout, dropout classique
        lancer("mlp_fmnist", 16, 50, frac, seed)
        lancer("cnn_fmnist", 4, 50, frac, seed)

## 2. Early dropout pendant 1 epoch sur 50 (10 runs)

In [ ]:
print("epochs de dropout :", round(0.02 * 50))    # doit afficher 1
for seed in [0, 1, 2, 3, 4]:
    lancer("mlp_fmnist", 16, 50, 0.02, seed)
    lancer("cnn_fmnist", 4, 50, 0.02, seed)

In [ ]:
attendus = {
    "graines neuves MLP": [fichier("mlp_fmnist", 16, 50, f, s) for f in [0, 1] for s in GRAINES_NEUVES],
    "graines neuves CNN": [fichier("cnn_fmnist", 4, 50, f, s) for f in [0, 1] for s in GRAINES_NEUVES],
    "1 epoch MLP": [fichier("mlp_fmnist", 16, 50, 0.02, s) for s in range(5)],
    "1 epoch CNN": [fichier("cnn_fmnist", 4, 50, 0.02, s) for s in range(5)],
    # références nécessaires pour l'analyse (déjà faites normalement)
    "sans dropout seeds 0-4": [fichier(c, L, 50, 0, s) for c, L in [("mlp_fmnist", 16), ("cnn_fmnist", 4)] for s in range(5)],
}
for nom, fichiers in attendus.items():
    print(f"{nom:24s} {sum(os.path.exists(f) for f in fichiers):2d} / {len(fichiers)}")

In [ ]:
!git config user.name "Felix-prjcts"
!git add resultats/
!git commit -m "runs complementaires : graines neuves 5-9 (regime) et early 1 epoch"

token = userdata.get("GITHUB_TOKEN")
!git push -q https://{token}@github.com/Felix-prjcts/early-dropout-study.git HEAD 2>&1 | sed "s/{token}/***/g"